In [ ]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

### important !!!!!!!           # cVoice13_tone_v2Mapping   cVoice13_2nd_withDiph
path = "/Users/evanchan19/Desktop/Thesis_Research/Mostafa_Phono_model/models/cVoice13_IpaAndTone_2nd"
processor = Wav2Vec2Processor.from_pretrained(path)
model = Wav2Vec2ForCTC.from_pretrained(path)

In [ ]:
from datasets import load_from_disk
import pandas as pd
import torch

data_path = "/Users/evanchan19/Desktop/Thesis_Research/Mostafa_Phono_model/speech_attribute/datasets/CommonVoice13_V4"

## Initial settings   ###  Importantant                         STEP 1)
file_path = "/Users/evanchan19/Desktop/Thesis_Research/Mostafa_Phono_model/speech_attribute/data/Mandarin_List_attr_IPAwithTone_V2.txt"

# Load the attribute mapping file                               STEP 2)
csv_file_path = "/Users/evanchan19/Desktop/Thesis_Research/Mostafa_Phono_model/speech_attribute/data/IPAwithTone_p2attr_V2.csv"
attribute_mapping = pd.read_csv(csv_file_path)

#### !!!!! VERY IMPORTANT                                       STEP 3)
which_type = "transcript_IPAwithTone"  # This is from the datasets itself !!!
which_sentence = "sentence"  # This is from the datasets itself !!!


### Chocie can be                   'tone'     or     'IPA'     STEP 4)
choice = "IpaAndTone"

data = load_from_disk(data_path)
data = data["test"]
data = data.select(range(50))
data

In [ ]:
import re


## Step 1
def load_attribute_list(file_path):
    try:
        with open(file_path) as f:
            list_att = f.read().splitlines()
            return list_att
    except FileNotFoundError:
        raise FileNotFoundError(f"The list of attribute file {file_path} is not found")


list_att = load_attribute_list(file_path)


## Step 2
def create_binary_groups(list_att):
    groups = []
    for att in list_att:
        binary_att = [f"p_{att}", f"n_{att}"]  # Each attribute could be +ve or -ve
        groups.append(binary_att)
    return groups


groups = create_binary_groups(list_att)


## Step 3
def get_att_group_indx_map(bTraining=True):
    # Get group ids dictionary
    group_ids = [sorted(processor.tokenizer.convert_tokens_to_ids(group)) for group in groups]
    if bTraining:
        group_ids = [dict([(x[1], x[0] + 1) for x in list(enumerate(g))]) for g in group_ids]
    else:
        group_ids = [dict([(x[0] + 1, x[1]) for x in list(enumerate(g))]) for g in group_ids]
    return group_ids


group_ids = get_att_group_indx_map(bTraining=False)


## Step 4) Decouple if needed
decouple_diph_file = "/Users/evanchan19/Desktop/Thesis_Research/Mostafa_Phono_model/speech_attribute/data/Diphthongs_Mandarin.csv"
with open(decouple_diph_file, "r", encoding="utf-8") as f:
    diphthongs_to_monophthongs_map = dict([(x.split(",")[0], " ".join(x.split(",")[1:])) for x in f.read().splitlines()])


def decouple_diphthongs(batch):
    # Compile a regular expression pattern to match any diphthong in the map
    pattern = r"|".join(re.escape(diph) for diph in diphthongs_to_monophthongs_map.keys())

    def replace_diphthong(phoneme_string):
        return re.sub(pattern, lambda x: diphthongs_to_monophthongs_map[x.group(0)], phoneme_string).split()

    return [phoneme for phoneme_string in batch for phoneme in replace_diphthong(phoneme_string)]

In [ ]:
def transpose_matrix(matrix):
    # Transpose the list of lists
    transposed_matrix = list(zip(*matrix))
    return transposed_matrix


def print_vertical(matrix):
    # Print each row of the matrix in a vertical format
    for row in matrix:
        print(" | ".join(row))


# Assuming `data` is a list of dictionaries with audio and reference sentences
prediction = []  ## A ordered speech attribute mapping of predcitions !!!!!!!
actual_sentence = []  ## A actual_sentence for checking !!!!!!!!!!
supposed_sentence = []
for item in data:
    audio_array = item["audio"]["array"]  # Extract the audio array
    sampling_rate = item["audio"]["sampling_rate"]  # Extract the sampling rate

    # Process the audio
    inputs = processor(audio_array, sampling_rate=sampling_rate, return_tensors="pt", padding=True)

    # Run the model inference
    with torch.no_grad():
        logits = model(inputs.input_values).logits

    item_output = []  # Store pred_temps for the current item only
    for i in range(len(group_ids)):
        mask = torch.zeros(logits.size()[2], dtype=torch.bool)
        mask[0] = True
        mask[list(group_ids[i].values())] = True
        logits_g = logits[:, :, mask]
        pred_ids = torch.argmax(logits_g, dim=-1)
        pred_ids = pred_ids.cpu().apply_(lambda x: group_ids[i].get(x, x))

        # Decode predictions and clean up
        temp = processor.batch_decode(pred_ids, spaces_between_special_tokens=True)[0]
        temp = temp.replace("p_", "")  # Perform replace during the temp stage
        item_output.append(temp.split())  # Collect the prediction parts for the current item

    # Append the item's transposed predictions to the global prediction
    transposed_item_output = transpose_matrix(item_output)
    prediction.append(transposed_item_output)  # Keep accumulating all items’ predictions in prediction

    actual_sentence.append(item[which_type].split())

    # Print actual_sentence and prediction
    # print("-" * 100)
    # print("Actual Sentence:", item[suppose_sentence])
    # print("Actual correct IPA phoneme/ tone:", item[which_type])
    # print("Model Prediction from what Speakers said:")
    # print_vertical(transposed_item_output)

if "tone" in which_type:
    actual_sentence = [[int(item) for item in sublist] for sublist in actual_sentence]

# Important
# is_decouple = True
# if (is_decouple):
#     actual_sentence = [decouple_diphthongs(item) for item in actual_sentence]

In [ ]:
from Levenshtein import editops
from collections import defaultdict
from typing import List, Dict, Tuple
from difflib import SequenceMatcher

# Function to compute similarity between two sets of attributes
method = "dice"  # or "dice"


def compute_similarity(set_a: set, set_b: set, method: str) -> float:
    if not set_a and not set_b:
        return 1.0  # Both empty sets are considered fully similar

    intersection = len(set_a & set_b)

    if method == "dice":
        total = len(set_a) + len(set_b)
        return 2 * intersection / total if total != 0 else 0.0
    elif method == "jaccard":
        union = len(set_a | set_b)
        return intersection / union if union != 0 else 0.0
    else:
        raise ValueError(f"Unsupported similarity method: {method}")


def attribute_matching(csv_dataFrame: pd.DataFrame, recognized_attribute_list: List[str], choice: str, method: str) -> Tuple[str, bool]:
    if csv_dataFrame.empty or not recognized_attribute_list:
        return "NoMandarinAttributeDetected!", True

    best_match = "NoMandarinAttributeDetected!"
    best_score = 0.0
    used_similarity = True

    recognized_set = set(recognized_attribute_list)

    for _, row in csv_dataFrame.iterrows():
        expected_set = {attr for attr, val in row.items() if val == 1}

        # Scenario 1: Exact match
        if recognized_set == expected_set:
            return row[choice], False

        # Scenario 2: Similarity match
        score = compute_similarity(recognized_set, expected_set, method=method)

        if score > best_score:
            best_score = score
            best_match = row[choice]

    return best_match, used_similarity


def align_sentence(actual: List[str], predicted: List[str]) -> Tuple[List[str], List[str]]:
    """
    Aligns two sequences (actual and predicted) using word-level SequenceMatcher.
    """
    matcher = SequenceMatcher(None, actual, predicted)
    aligned_actual = []
    aligned_pred = []

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "equal":
            aligned_actual.extend(actual[i1:i2])
            aligned_pred.extend(predicted[j1:j2])
        elif tag == "replace":
            for i, j in zip(range(i1, i2), range(j1, j2)):
                aligned_actual.append(actual[i])
                aligned_pred.append(predicted[j])
            # Handle unequal lengths
            if (i2 - i1) > (j2 - j1):
                for i in range((i2 - i1) - (j2 - j1)):
                    aligned_actual.append(actual[i1 + (j2 - j1) + i])
                    aligned_pred.append("∅")
            elif (j2 - j1) > (i2 - i1):
                for j in range((j2 - j1) - (i2 - i1)):
                    aligned_actual.append("∅")
                    aligned_pred.append(predicted[j1 + (i2 - i1) + j])
        elif tag == "delete":
            for i in range(i1, i2):
                aligned_actual.append(actual[i])
                aligned_pred.append("∅")
        elif tag == "insert":
            for j in range(j1, j2):
                aligned_actual.append("∅")
                aligned_pred.append(predicted[j])

    return aligned_pred, aligned_actual


# 更新后的 compute_metrics
def compute_metrics(prediction: str, actual: str) -> Tuple[List[str], List[str]]:
    pred = prediction.split()
    act = actual.split()

    aligned_pred, aligned_act = align_sentence(act, pred)

    metrics_result = []
    diagnosed_result = []

    for p, a in zip(aligned_pred, aligned_act):
        if p == a:
            metrics_result.append("TA")
            diagnosed_result.append("CD")
        else:
            metrics_result.append("FR")
            diagnosed_result.append("WD")

    return metrics_result, diagnosed_result


def compute_metrics_percentage(metrics: List, diagnosed: List) -> Tuple[float, float, float, float]:
    total = len(metrics)

    if total == 0:
        return 0.0, 0.0, 0.0, 0.0

    TA = round(metrics.count("TA") / total, 2) * 100
    FR = round(metrics.count("FR") / total, 2) * 100
    CD = round(diagnosed.count("CD") / total, 2) * 100
    WD = round(diagnosed.count("WD") / total, 2) * 100

    return TA, FR, CD, WD


# This is one-to-one matching
def compute_error_type(prediction: List, supposed: List):
    pred_temp = prediction.split()
    supposed_temp = supposed.split()
    result = []

    for i in range(len(pred_temp)):
        if pred_temp[i] == supposed_temp[i]:
            result.append("ok")
            continue

        if pred_temp[i] == "Unknown":
            result.append("Not CN word")
            continue

        if len(pred_temp[i]) == len(supposed_temp[i]):
            result.append("Substitution")
        elif len(pred_temp[i]) > len(supposed_temp[i]):
            result.append("Insertion")
        else:
            result.append("Deletion")

    return result


# Only useful in IpaAndTone together
def detail_checking(reference, predicted):
    result = {"phoneme_errors": {}, "tone_errors": {}}
    if not reference or not predicted:
        return result

    pred_phoneme = predicted[:-1] if predicted and predicted[-1].isdigit() else predicted
    pred_tone = predicted[-1] if predicted and predicted[-1].isdigit() else None
    rec_phoneme = reference[:-1] if reference and reference[-1].isdigit() else reference
    rec_tone = reference[-1] if reference and reference[-1].isdigit() else None

    if pred_phoneme != rec_phoneme:
        result["phoneme_errors"][rec_phoneme] = result["phoneme_errors"].get(rec_phoneme, 0) + 1
    elif pred_tone != rec_tone:
        result["tone_errors"][rec_tone] = result["tone_errors"].get(rec_tone, 0) + 1

    return result


def compute_levenshtein_metrics(ref: List[str], pred: List[str]) -> Tuple[Dict, List[str]]:
    cm = defaultdict(lambda: defaultdict(int))
    ops = editops(ref, pred)

    # 每个 ref_index 可能有多个 insert，在前面插入时不能直接操作原始列表
    error_map = defaultdict(list)

    for op, ref_i, pred_i in ops:
        if op == "insert":
            msg = f"ins({pred_i}:'{pred[pred_i]}')"
            error_map[ref_i].append(msg)
            cm["insertions"][pred[pred_i]] += 1
        elif op == "delete":
            msg = f"del({ref_i}:'{ref[ref_i]}')"
            error_map[ref_i].append(msg)
            cm["deletions"][ref[ref_i]] += 1
        elif op == "replace":
            msg = f"sub({ref_i}:'{ref[ref_i]}' -> '{pred[pred_i]}')"
            error_map[ref_i].append(msg)
            cm["substitutions"][(ref[ref_i], pred[pred_i])] += 1

    # 构造 error_list：每个 token 按照 ref 的顺序走，如果某处有插入，也加进去
    error_list = []
    for i in range(len(ref)):
        if i in error_map:
            for msg in error_map[i]:
                if msg.startswith("ins"):
                    error_list.append(msg)
        if i in error_map and any(m.startswith("del") or m.startswith("sub") for m in error_map[i]):
            # 如果有 delete 或 substitution，优先取第一个非-insert的
            main_error = next(m for m in error_map[i] if not m.startswith("ins"))
            error_list.append(main_error)
        else:
            error_list.append("ok")

    # 补充末尾如果还有 insert 操作（例如插入到 ref 最末尾）
    if len(ref) in error_map:
        for msg in error_map[len(ref)]:
            if msg.startswith("ins"):
                error_list.append(msg)

    return cm, error_list


def pretty_print_leveshtein_errors(errors: List[str]):
    for error in errors:
        print(error)


# def align_sentence(ref: List[str], target: List[str], errors: List[str]):
#    # Align the reference and prediction with error tags
#     aligned_ref = []
#     ref_index = 0       # Track position in reference
#     del_ins_count = 0

#     for error in errors:
#         if error.startswith("ins"):  # Insertion case
#             aligned_ref.append("ins")  # Add the phoneme from reference
#             del_ins_count += 1
#         elif error.startswith("del"):  # Deletion case
#             aligned_ref.append("del")  # Placeholder for deletion
#             del_ins_count += 1
#         else:
#             if ref_index < len(ref):
#                 aligned_ref.append(ref[ref_index])
#             ref_index += 1  # Move to the next reference element

#         if len(ref) + del_ins_count == len(target):
#             return ' '.join(aligned_ref) + ' ' + ' '.join(ref[ref_index:])

#     return ' '.join(aligned_ref)


def check_prediction(predicted_attributes, reference, choice, detailCheck: bool):
    # step 1) First check the choice of prediction
    if choice == "tone":
        name = "Phoneme_PinyinTon"
    elif choice == "IPA":
        name = "Phoneme_ipaDragon"
    elif choice == "IpaAndTone":
        name = "Phoneme_ipaDragon"
    else:
        return "wrong function input"

    # Then Filter the mapping for the current phoneme tone
    expected_attributes = attribute_mapping[attribute_mapping[name] == reference]
    ## Optional -> Show the refernce in Pinyin/tone
    # reference = expected_attributes[optional].iloc[0]

    expected_attributes = expected_attributes.iloc[0].iloc[3:]

    # Step 2) Run the checking
    expected_attributes = [attribute for attribute, expected in expected_attributes.items() if expected == 1]

    predicted_set = set(predicted_attributes)
    correct_set = set(expected_attributes)

    check = correct_set.issubset(predicted_set)
    wrong_list = list(correct_set - predicted_set)
    extra_list = list(predicted_set - correct_set)

    # Step 3) Convert the prediction into Phoneme/tone
    recognized_phoneme, is_similiar = attribute_matching(attribute_mapping, predicted_attributes, name, method)

    if check and not is_similiar:
        print(f"✅ Correct Pronunciation!")
    elif check and is_similiar:
        print(f"✅ Similar Pronunciation Detected!")
    else:
        print("❌ Wrong Pronunciation Detected!")

    print(f"Phoneme/Tone: {reference}")

    # Print the most similar pronunciation if available
    if is_similiar:
        print(f"Most Similar Pronunciation: {recognized_phoneme}\n")
    else:
        print(f"Recognized as: {recognized_phoneme}\n")

    # Step 4: Print detailed attribute analysis if requested
    if detailCheck and is_similiar:
        print("Details:")
        print(f"  - Missing Attributes: {', '.join(wrong_list) if wrong_list else 'None'}")
        print(f"  - Extra Attributes: {', '.join(extra_list) if extra_list else 'None'}")
        print(f"  - Your Pronunciation Attributes: {', '.join(predicted_attributes)}")
        print(f"  - Expected Attributes: {', '.join(expected_attributes)}\n")
        detail_checking(reference, recognized_phoneme)

    return recognized_phoneme, reference

In [ ]:
## Special case
import re


# Function to split phonemes like "wei4" → ["w", "e", "i", "4"]
def split_phoneme(phoneme):
    match = re.match(r"([a-zA-Zɕʂʐŋɔɑɨʊioɛɤə]+)([0-5]?)", phoneme)
    if match:
        core = list(match.group(1))
        tone = [match.group(2)] if match.group(2) else []
        return core + tone
    return [phoneme]


# Initialize counters and storage for sentences
i = 0
result = {}

total_TA = 0
total_FR = 0
total_CD = 0
total_WD = 0

print("=" * 20 + " The Evaluation stat now " + "=" * 20)
print("=" * 65 + "\n")

for sentence in prediction:
    actual = []
    pred = []

    raw_actual_sentence = actual_sentence[i]

    for j, item_list in enumerate(sentence):
        predicted_attributes = [item for item in item_list if "n_" not in item]

        if j >= len(raw_actual_sentence):
            break

        ref_phoneme = raw_actual_sentence[j]
        out, ref = check_prediction(predicted_attributes, ref_phoneme, choice, True)

        actual.extend(split_phoneme(ref))
        pred.extend(split_phoneme(out))

    # Skip sentence if actual and pred length difference is 2 or more
    if abs(len(actual) - len(pred)) >= 2:
        print(f"Skipping sentence {i + 1} due to length mismatch (Actual: {len(actual)}, Pred: {len(pred)})\n")
        i += 1
        continue

    actual_str = " ".join(actual)
    pred_str = " ".join(pred)

    print("=" * 20 + f" SUMMARY: Sentence {i + 1} " + "=" * 20)
    print(f"Actual:                             {actual_str}")
    print(f"Model Predictions:                  {pred_str}")

    metrics, diagnosed = compute_metrics(pred_str, actual_str)
    TA, FR, CD, WD = compute_metrics_percentage(metrics, diagnosed)

    print(f"TA percentage:             {TA:.2f}%, FR {FR:.2f}%")
    print(f"CD percentage:             {CD:.2f}%, WD {WD:.2f}%")

    total_TA += TA
    total_FR += FR
    total_CD += CD
    total_WD += WD

    print("\nNew sentence evaluation starts now\n")
    i += 1

In [ ]:
# Print the total calculations
num_sentences = i
avg_TA = total_TA / (total_TA + total_FR) * 100
avg_FR = total_FR / (total_TA + total_FR) * 100
avg_CD = total_CD / (total_CD + total_WD) * 100
avg_WD = total_WD / (total_CD + total_WD) * 100

## Final Confusion Amtrix
print("=" * 30 + " FINAL CONFUSION MATRIX " + "=" * 30)
print(f"Total Sentences Evaluated: {num_sentences}")
print(f"Total MDD Metrics:   TA {total_TA}, FR {total_FR}, CD {total_CD}, WD {total_WD}")
print(f"Average MDD %:       TA {avg_TA:.2f}%, FR {avg_FR:.2f}%, CD {avg_CD:.2f}%, WD {avg_WD:.2f}%")
print("=" * 80 + "\n")